[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-IA_Ed2026/blob/main/meca-ia/C15/E15-1_Pieces_8x8.ipynb)

# Contrôle de rondelles par caméra : conforme ou non conforme

## Le contexte

En sortie de ligne, une caméra photographie chaque rondelle usinée, en 8 sur 8 pixels. Une rondelle est conforme quand son diamètre, son perçage, son centrage et sa forme respectent les tolérances. Le contrôle visuel manuel est lent et varie d'un opérateur à l'autre ; le service qualité voudrait que la caméra trie seule. Un échantillon de 4 000 rondelles a été photographié et classé par un contrôleur. C'est le jeu du chapitre précédent, avec une question plus simple : bonne ou mauvaise.

La grandeur à prédire est une catégorie à deux valeurs, conforme ou non : c'est un problème de classification.

Défi : prédire `conforme` et obtenir une exactitude supérieure à celle du modèle de départ sur la base de test. Ce modèle de départ est le réseau dense de scikit-learn du chapitre précédent, aux réglages par défaut ; il atteint une exactitude de 0,75. À vous de faire mieux avec un réseau convolutif de Keras. La règle de découpage est fixée plus bas, et c'est celle du défi précédent. La base de test sert à mesurer, jamais à apprendre ni à régler.

## Les données

Fichier `MecaIA-Pieces_8x8.xlsx`, dans le dossier `donnees` du dépôt : 4 000 images, 66 colonnes. Une ligne est l'image 8 × 8 d'une pièce vue par la caméra de contrôle. C'est le jeu du chapitre précédent, avec l'autre cible.

| Colonne | Signification | Unité |
|---|---|---|
| `p00` à `p63` | pixels de l'image, lus ligne par ligne : `p00` en haut à gauche, `p07` en haut à droite, `p63` en bas à droite | niveau de gris, 0 noir et 255 blanc |
| `conforme` | cible : 1 si la pièce respecte toutes les tolérances, 0 sinon | 0 ou 1 |
| `type_defaut` | type de défaut, première cible du jeu, écartée ici : la pièce est conforme exactement quand il vaut `aucun`, la donner au modèle reviendrait à lui donner la réponse ; elle sert seulement à répartir le tirage du test, pour que cette base soit celle du défi précédent | catégorie |

Origine : jeu simulé, rendu géométrique d'une rondelle claire sur fond sombre, avec des défauts de dimension et de forme, une position qui varie dans le champ, un éclairage variable et le bruit du capteur. Il enseigne une méthode, et aucune conclusion de métrologie n'en découle.

### Étape 1 : charger le fichier et regarder sa forme

On lit le classeur, puis on vérifie ses dimensions, ses cases vides et la répartition de la cible avant toute modélisation : la forme d'un fichier se lit sur le fichier entier. Le dossier des données est cherché depuis le dossier du notebook, puis depuis un clone du dépôt ; ouvert seul, par exemple dans Colab, le notebook télécharge une fois le jeu depuis le dépôt public.

In [ ]:
from pathlib import Path
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn import set_config

set_config(display="text")
GRAINE = 42
FICHIER = "MecaIA-Pieces_8x8.xlsx"
URL_JEU = "https://raw.githubusercontent.com/philippe-amuzuga/Meca-IA_Ed2026/main/donnees/MecaIA-Pieces_8x8.xlsx"

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-IA_Ed2026", "donnees")]
dossier = next((d for d in candidats if (d / FICHIER).is_file()), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique du jeu depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    try:
        urllib.request.urlretrieve(URL_JEU, dossier / FICHIER)
    except OSError as erreur:
        raise FileNotFoundError(
            "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
df = pd.read_excel(dossier / FICHIER)

cible = "conforme"
pixels = [f"p{k:02d}" for k in range(64)]
print("dimensions :", df.shape)
print("cases vides :", int(df.isna().sum().sum()))
print(df[cible].value_counts().to_dict())
df[pixels[:5] + [cible]].head(3)

Lecture du résultat : 4 000 images, 66 colonnes, aucune case vide. Les pièces conformes et non conformes sont à parts égales : une exactitude de 50 % est donc ce que donne le hasard, et une exactitude se lit sans précaution particulière sur la répartition.

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas le dossier `donnees/` du dépôt. La cellule suivante ne télécharge rien : elle cherche ce dossier (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-IA_Ed2026`), vérifie la présence du classeur du défi et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez le dossier `donnees` à la racine de l'espace de travail et téléversez-y le classeur listé, pris dans le dépôt.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
from pathlib import Path

JEUX_LUS = ["MecaIA-Pieces_8x8.xlsx"]                  # le classeur lu par ce notebook


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-IA_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees = chercher("donnees")
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Reproductibilité

Un réseau de neurones part de poids tirés au hasard et lit ses exemples dans un ordre tiré au hasard. La cellule suivante fixe toutes les graines, demande des opérations de calcul déterministes, impose un seul fil de calcul et un jeu d'instructions donné. Avec elle, deux exécutions du notebook sur le même ordinateur donnent les mêmes poids et le même score. Chaque cellule qui construit un réseau rappelle la graine dans sa première ligne : on peut la rejouer seule sans changer le résultat. Sur un autre ordinateur, ou sur un processeur qui mêle deux types de cœurs, les arrondis peuvent différer et le score changer à la troisième décimale.

In [ ]:
# Réseaux de neurones : mêmes résultats à chaque exécution.
# À placer après la cellule de lecture (qui définit GRAINE),
# avant tout usage de TensorFlow.
import os

# Lues à l'import de TensorFlow : journal réduit, oneDNN limité
# au produit matriciel, jeu d'instructions fixé.
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
os.environ["ONEDNN_MAX_CPU_ISA"] = "AVX2"

import numpy as np
import tensorflow as tf
import keras

# Graines, opérations déterministes et un seul fil de calcul.
keras.utils.set_random_seed(GRAINE)
tf.config.experimental.enable_op_determinism()
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.threading.set_inter_op_parallelism_threads(1)

print("TensorFlow", tf.__version__, "| Keras", keras.__version__,
      "| NumPy", np.__version__)

### Étape 2 : mettre le test de côté

Un quart des images est mis de côté avant tout apprentissage, tout réglage et toute comparaison avec la cible. Le tirage est stratifié sur le type de défaut, comme dans le défi précédent : les deux défis partagent ainsi la même base de test, sans fuite possible puisque chacun n'apprend que sur les lignes d'apprentissage. La graine 42 fixe le tirage : chacun obtient la même base de test, et les scores se comparent. Dans l'apprentissage restant, un cinquième est réservé à la validation : il servira à comparer les réseaux et à décider quand arrêter l'entraînement, jamais le test.

In [ ]:
from sklearn.model_selection import train_test_split

X = df[pixels]
y = df[cible]
strate = df["type_defaut"]                  # répartit le tirage, n'entre jamais dans le modèle
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=GRAINE, stratify=strate)
X_fit, X_val, y_fit, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=GRAINE, stratify=strate[X_train.index])
print("apprentissage :", len(X_fit), "images")
print("validation :", len(X_val), "images")
print("test :", len(X_test), "images")

### Étape 3 : regarder les images de l'apprentissage

La figure montre six images tirées au hasard de l'apprentissage pour chaque classe : en haut les pièces conformes, en bas les pièces non conformes. Le tirage ne touche que `X_train`.

In [ ]:
rng = np.random.default_rng(GRAINE)
fig, axes = plt.subplots(2, 6, figsize=(6.4, 2.4))
for i, (valeur, nom) in enumerate([(1, "conforme"), (0, "non conforme")]):
    for j, position in enumerate(rng.choice(X_train.index[y_train == valeur], 6, replace=False)):
        axes[i, j].imshow(X_train.loc[position].to_numpy().reshape(8, 8), cmap="gray", vmin=0, vmax=255)
        axes[i, j].set_xticks([])
        axes[i, j].set_yticks([])
    axes[i, 0].set_ylabel(nom, rotation=0, ha="right", va="center", fontsize=9)
plt.tight_layout()
plt.show()
moyenne_classe = {valeur: X_train[y_train == valeur].mean().to_numpy().reshape(8, 8) for valeur in (0, 1)}
decalage = np.abs(moyenne_classe[1] - moyenne_classe[0]).max()
print(f"écart maximal entre l'image moyenne des conformes et celle des non conformes : {decalage:.0f} niveaux de gris sur 255")

Lecture du résultat : à l'œil, une rondelle conforme et une rondelle non conforme se ressemblent. Le défaut est une affaire de demi-pixel, et la rondelle bouge dans le champ d'une image à l'autre : l'image moyenne de chaque classe est floue, et les deux moyennes ne diffèrent que de 7 niveaux de gris au plus. Aucun pixel pris isolément ne dit si la pièce est bonne : c'est la forme, c'est-à-dire la relation entre pixels voisins, qui compte.

### Étape 4 : construire le modèle de départ

Le modèle de départ est le réseau dense de scikit-learn du chapitre précédent, aux réglages par défaut : une couche cachée de 100 neurones, 200 époques au plus. Un pipeline ramène d'abord chaque pixel à une moyenne nulle et à un écart type de 1, puis ajuste le réseau sur `X_train` seul.

In [ ]:
# Votre code :

### Étape 5 : mesurer sur la base de test

Le modèle prédit les images qu'il n'a jamais vues. Deux indicateurs sont lus : l'exactitude, part des images bien classées, et le rappel des pièces non conformes, part des pièces défectueuses que le modèle arrête. Un témoin sans apprentissage, qui répond toujours la même classe, donne le point de comparaison le plus bas.

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, recall_score

y_pred = modele.predict(X_test)
score_depart = accuracy_score(y_test, y_pred)
temoin = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"exactitude : {score_depart:.3f}")
print(f"rappel des pièces non conformes : {recall_score(y_test, y_pred, pos_label=0):.3f}")
print(f"exactitude du témoin (toujours la classe la plus fréquente) : {accuracy_score(y_test, temoin.predict(X_test)):.3f}")
print(f"époques effectuées : {modele[-1].n_iter_} sur {modele[-1].max_iter} permises")

Lecture du résultat : l'exactitude est de 0,75 et le réseau arrête 71 % des pièces non conformes. Le témoin, qui répond toujours la même classe, a une exactitude de 0,50, celle du hasard. Le réseau a effectué 200 époques sur les 200 permises : il s'est arrêté faute d'époques, sans avoir convergé. C'est cette exactitude, mesurée sur des images que le modèle n'a jamais vues, que l'on a le droit d'annoncer.

## Pour aller plus loin : un réseau convolutif de Keras

Le modèle de départ de l'étape 4 reste le score de référence du défi. Les cellules suivantes montrent comment le dépasser sans regarder le test pour choisir. Un réseau dense relie chaque pixel à chaque neurone et ignore la place du pixel dans l'image. Une couche de convolution fait glisser sur l'image une petite fenêtre de 3 sur 3 pixels, avec les mêmes poids partout : elle détecte un motif local, où qu'il se trouve. La première cellule prépare des images de 8 sur 8 pour Keras, avec l'échelle apprise sur la partie d'apprentissage seule. La deuxième compare, sur la validation, un réseau dense de Keras et trois réseaux convolutifs, avec un arrêt précoce. La troisième compare leurs courbes d'apprentissage. La quatrième regarde, sur la validation, quels types de défaut chaque réseau laisse passer. La dernière mesure le réseau retenu une seule fois sur le test. Les valeurs sont lues dans les sorties imprimées.

In [ ]:
# Votre code :

In [ ]:
# Votre code :

In [ ]:
# Votre code :

fig, ax = plt.subplots(figsize=(6.4, 3.6))
for nom, style, gris in [("dense 128-64", "--", "0.55"), (nom_retenu, "-", "0.0")]:
    ax.plot(range(1, len(histoires[nom].history["val_accuracy"]) + 1), histoires[nom].history["val_accuracy"],
            linestyle=style, color=gris, label=nom)
ax.set(xlabel="époque", ylabel="exactitude de validation", ylim=(0.5, 1.0))
ax.legend(frameon=False)
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# Votre code :

In [ ]:
# Votre code :

Lecture du résultat : sur la validation, le réseau de scikit-learn par défaut atteint une exactitude de 0,72. Le réseau dense de Keras (16 706 paramètres), aux entrées mises à l'échelle et arrêté tôt, atteint déjà 0,86 : une grande part du gain tient à cette préparation, avant toute convolution. Les trois réseaux convolutifs vont de 0,81 à 0,89 ; la validation retient « convolution 32-64 » (35 394 paramètres), 0,03 au-dessus du réseau dense. Cet avantage est modeste et se concentre sur certains défauts : sur la validation, le réseau retenu retrouve 85 % des ovalisations contre 69 % pour le réseau dense, et 98 % des décentrages contre 90 %. Les ébréchures restent mal vues par les deux réseaux (40 % pour le réseau retenu, 45 % pour le réseau dense) : un défaut d'un demi-pixel à la rive d'une image de 8 pixels de côté se confond avec le bruit. Mesuré une seule fois sur le test, le réseau retenu atteint une exactitude de 0,89, soit 0,14 de plus que le modèle de départ.

## Le score de référence

Le score publié est l'exactitude du modèle de départ de l'étape 4 sur la base de test. L'empreinte identifie cette base de test : si la vôtre diffère, le découpage n'est pas le même et les scores ne se comparent pas.

In [ ]:
import hashlib

score = accuracy_score(y_test, modele.predict(X_test))
positions = ",".join(str(i) for i in sorted(X_test.index))
print("EMPREINTE_TEST =", hashlib.sha256(positions.encode("ascii")).hexdigest())
print("SCORE_REFERENCE accuracy = %.2f" % score)

## Synthèse

1. Un réseau dense lit 64 pixels indépendants ; une convolution fait glisser la même fenêtre de 3 sur 3 pixels sur toute l'image, avec les mêmes poids partout, et détecte un motif où qu'il se trouve.
2. Aux réglages par défaut, le réseau de scikit-learn atteint une exactitude de 0,75 sur le test ; le réseau convolutif retenu sur la validation atteint 0,89.
3. La comparaison des architectures se fait sur une validation prise dans l'apprentissage, avec un arrêt précoce qui rend les poids de la meilleure époque ; le test ne sert qu'une fois.
4. Une part du gain vient de la préparation (échelle, arrêt précoce), le reste de la convolution, qui retrouve mieux les ovalisations et les décentrages ; l'ébréchure reste le défaut que les deux réseaux voient mal.
5. La graine est rappelée dans la cellule qui construit chaque réseau : on la rejoue seule sans changer le résultat.